In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# ====================== 全局样式统一配置 ======================
# 解决中文乱码
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
# 高清输出 + 统一尺寸
plt.rcParams['figure.dpi'] = 150
plt.rcParams['figure.figsize'] = (8, 4.5)
# 统一商务配色
MAIN_PALETTE = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

# ====================== 自动创建输出目录 ======================
os.makedirs('output/charts', exist_ok=True)

# ====================== 读取所有计算结果 ======================
# 1. 流量数据
df_daily_flow = pd.read_csv('output/data/daily_pv_uv.csv',
                            parse_dates=['date'], encoding='utf-8-sig')
df_hour_flow = pd.read_csv('output/data/hour_pv.csv', encoding='utf-8-sig')

# 2. 转化数据
df_funnel = pd.read_csv('output/data/conversion_funnel.csv', encoding='utf-8-sig')
df_category_conv = pd.read_csv('output/data/category_conv.csv', encoding='utf-8-sig')

# 3. 用户留存数据（同期群）
df_retention = pd.read_csv('output/data/user_retention.csv',
                           index_col='first_month', encoding='utf-8-sig')

# 4. RFM用户分层数据
df_rfm = pd.read_csv('output/data/rfm_segment.csv', encoding='utf-8-sig')

# 5. 商品数据
df_product_top = pd.read_csv('output/data/product_sales_cnt.csv', encoding='utf-8-sig')
df_category_sales = pd.read_csv('output/data/category_sales.csv', encoding='utf-8-sig')

print(" 所有数据读取完成")


 所有数据读取完成


In [2]:
'''--------- ---------'''
'''---------1. 流量规模指标图表 ---------'''
# 1.1 每日PV/UV趋势折线图（双Y轴）
def plot_daily_pv_uv():
    fig, ax1 = plt.subplots()

    # 左轴 - PV
    ax1.plot(df_daily_flow['date'], df_daily_flow['pv'],
             color=MAIN_PALETTE[0], linewidth=2, label='PV')
    ax1.set_ylabel('访问量(PV)', color=MAIN_PALETTE[0], fontsize=10)
    ax1.tick_params(axis='y', labelcolor=MAIN_PALETTE[0])
    ax1.tick_params(axis='x', rotation=45, labelsize=8)

    # 右轴 - UV
    ax2 = ax1.twinx()
    ax2.plot(df_daily_flow['date'], df_daily_flow['uv'],
             color=MAIN_PALETTE[1], linewidth=2, label='UV')
    ax2.set_ylabel('用户数(UV)', color=MAIN_PALETTE[1], fontsize=10)
    ax2.tick_params(axis='y', labelcolor=MAIN_PALETTE[1])

    # 标题与图例
    plt.title('每日PV/UV流量趋势', fontsize=12, pad=15)
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=8)

    fig.tight_layout()
    plt.savefig('output/charts/流量_每日PVUV趋势.png', bbox_inches='tight')
    plt.close()
    print(" 已生成：流量_每日PVUV趋势.png")

# 1.2 时段PV分布柱状图
def plot_hour_pv():
    plt.figure()
    sns.barplot(x='hour', y='pv', data=df_hour_flow,
                color=MAIN_PALETTE[0], alpha=0.85)

    plt.title('全天各时段PV流量分布', fontsize=12, pad=15)
    plt.xlabel('小时', fontsize=10)
    plt.ylabel('PV访问量', fontsize=10)
    plt.grid(axis='y', linestyle='--', alpha=0.3)

    plt.tight_layout()
    plt.savefig('output/charts/流量_时段PV分布.png', bbox_inches='tight')
    plt.close()
    print(" 已生成：流量_时段PV分布.png")
'''---------转化指标图表 ---------'''
# 2.1 整体转化漏斗图
def plot_conversion_funnel():
    fig = px.funnel(df_funnel,
                    x='user_count',
                    y='stage',
                    title='用户全链路转化漏斗',
                    color_discrete_sequence=[MAIN_PALETTE[0]])

    fig.update_layout(
        width=800,
        height=500,
        title_x=0.5,
        font=dict(size=12)
    )

    fig.write_image('output/charts/转化_整体转化漏斗.png', scale=2)
    print("已生成：转化_整体转化漏斗.png")

# 2.2 分品类转化率对比柱状图（横向）
def plot_category_conversion():
    # 过滤转化率为0的品类，按转化率升序排列
    df_sorted = df_category_conv[df_category_conv['convert_rate'] > 0]\
                    .sort_values('convert_rate', ascending=True)

    plt.figure(figsize=(8, 5))
    bars = plt.barh(df_sorted['category_id'].astype(str),
                    df_sorted['convert_rate'],
                    color=MAIN_PALETTE[0], alpha=0.85)

    # 柱子上标注百分比
    for bar in bars:
        width = bar.get_width()
        plt.text(width + 0.002, bar.get_y() + bar.get_height()/2,
                 f'{width:.2%}', va='center', fontsize=9)

    plt.title('各品类支付转化率对比', fontsize=12, pad=15)
    plt.xlabel('支付转化率', fontsize=10)
    plt.ylabel('品类ID', fontsize=10)
    plt.grid(axis='x', linestyle='--', alpha=0.3)

    plt.tight_layout()
    plt.savefig('output/charts/转化_分品类转化率对比.png', bbox_inches='tight')
    plt.close()
    print(" 已生成：转化_分品类转化率对比.png")
'''---------3. 用户分析图表 ---------'''
# 3.1 同期群留存热力图
def plot_retention_heatmap():
    plt.figure(figsize=(10, 5))
    sns.heatmap(df_retention,
                annot=True,
                fmt='.1%',
                cmap='YlGnBu',
                annot_kws={'size': 10},
                linewidths=0.5,
                cbar_kws={'label': '留存率'})

    plt.title('用户同期群留存热力图（按月）', fontsize=12, pad=15)
    plt.xlabel('距首次活跃月数', fontsize=10)
    plt.ylabel('首次活跃月份', fontsize=10)

    plt.tight_layout()
    plt.savefig('output/charts/用户_同期群留存热力图.png', bbox_inches='tight')
    plt.close()
    print(" 已生成：用户_同期群留存热力图.png")

# 3.2 RFM用户价值分层环形图
def plot_rfm_segment():
    plt.figure()
    plt.pie(df_rfm['用户数'],
            labels=df_rfm['user_type'],
            autopct='%1.1f%%',
            colors=MAIN_PALETTE[:len(df_rfm)],
            wedgeprops=dict(width=0.3, edgecolor='white'),
            textprops={'fontsize': 9})

    plt.title('RFM用户价值分层占比', fontsize=12, pad=15)

    plt.tight_layout()
    plt.savefig('output/charts/用户_RFM分层占比.png', bbox_inches='tight')
    plt.close()
    print(" 已生成：用户_RFM分层占比.png")

'''---------4. 商品分析图表 ---------'''
# 4.1 商品销量TOP10柱状图
def plot_product_top10():
    # 取销量前10并倒序，方便横向柱状图从上到下递减
    df_top = df_product_top.sort_values('sale_count', ascending=True).tail(10)

    plt.figure(figsize=(8, 5))
    plt.barh(df_top['product_id'].astype(str),
             df_top['sale_count'],
             color=MAIN_PALETTE[2], alpha=0.85)

    plt.title('商品销量TOP10', fontsize=12, pad=15)
    plt.xlabel('销量', fontsize=10)
    plt.ylabel('商品ID', fontsize=10)
    plt.grid(axis='x', linestyle='--', alpha=0.3)

    plt.tight_layout()
    plt.savefig('output/charts/商品_销量TOP10.png', bbox_inches='tight')
    plt.close()
    print(" 已生成：商品_销量TOP10.png")

# 4.2 品类销售额占比饼图
def plot_category_share():
    plt.figure()
    plt.pie(df_category_sales['sale_count'],
            labels=df_category_sales['category_id'].astype(str),
            autopct='%1.1f%%',
            colors=MAIN_PALETTE,
            textprops={'fontsize': 9})

    plt.title('各品类销量占比', fontsize=12, pad=15)

    plt.tight_layout()
    plt.savefig('output/charts/商品_品类销量占比.png', bbox_inches='tight')
    plt.close()
    print(" 已生成：商品_品类销量占比.png")

'''---------5. 一键生成所有图表 ---------'''
def generate_all_charts():
    print("===== 开始生成所有图表 =====")

    # 流量模块
    plot_daily_pv_uv()
    plot_hour_pv()

    # 转化模块
    plot_conversion_funnel()
    plot_category_conversion()

    # 用户模块
    plot_retention_heatmap()
    plot_rfm_segment()

    # 商品模块
    plot_product_top10()
    plot_category_share()

    print("===== 所有图表生成完成！请查看 output/charts/ 目录 =====")

# 执行一键生成
generate_all_charts()


===== 开始生成所有图表 =====
 已生成：流量_每日PVUV趋势.png
 已生成：流量_时段PV分布.png
已生成：转化_整体转化漏斗.png
 已生成：转化_分品类转化率对比.png
 已生成：用户_同期群留存热力图.png
 已生成：用户_RFM分层占比.png
 已生成：商品_销量TOP10.png
 已生成：商品_品类销量占比.png
===== 所有图表生成完成！请查看 output/charts/ 目录 =====
